In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import RAW_DATA, INTERIM_DATA

In [2]:
option = pd.read_parquet(RAW_DATA / 'spy-option.parquet')

option['expiration'] = pd.to_datetime(option['expiration'])
option['quote_date'] = pd.to_datetime(option['quote_date'])

print(option.columns.to_list(), f'\n\n{option.shape}')

['quote_date', 'symbol', 'expiration', 'type', 'strike', 'last', 'bid', 'ask', 'change', 'percent_change', 'volume', 'open_interest', 'implied_volatility', 'delta', 'gamma', 'theta', 'vega', 'rho'] 

(22025816, 18)


## Call option data

In [3]:
call = option[option['type'] == 'call'].copy()
call.shape

(11012971, 18)

## 0- Removing nan values

In [4]:
call_0 = call.copy()

before = len(call_0)

num_cols = call_0.select_dtypes(include='number').columns
call_0[num_cols] = call_0[num_cols].replace([-9999999, -999999.0, 9999999, 999999.0], pd.NA)
call_0 = call_0.dropna()

after = len(call_0)
print(f'Removed observations: {before - after:,}  ({(before-after)/before*100:.1f}% of data)')

Removed observations: 70,734  (0.6% of data)


## 1- Basic data cleaning

In [5]:
call_1 = call_0.copy()

before = len(call_1)

call_1 = call_1[
    (call_1['bid'] > 0) &
    (call_1['ask'] > call_1['bid']) &
    (call_1['strike'] > 0) &
    (call_1['implied_volatility'] > 0) &
    (call_1['expiration'] > call_1['quote_date'])
].copy()

call_1['mid'] = (call_1['bid'] + call_1['ask']) / 2
call_1['spread'] = call_1['ask'] - call_1['bid']
call_1['rel_spread'] = call_1['spread'] / call_1['mid']

call_1 = call_1[call_1['mid'] > 0].copy()

after = len(call_1)
print(f'Removed observations: {before - after:,}  ({(before-after)/before*100:.1f}% of data)')

Removed observations: 975,190  (8.9% of data)


## 2- Strike smoothness 
- compute finite differences

In [6]:
call_2 = call_1.copy()
call_2 = call_2.sort_values(['quote_date', 'expiration', 'strike'])

group_cols = ['quote_date', 'expiration']

call_2['iv_diff_1'] = call_2.groupby(group_cols)['implied_volatility'].diff()
call_2['iv_diff_2'] = call_2.groupby(group_cols)['iv_diff_1'].diff()

def median_(x):
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    return np.median(np.abs(x))

def max_(x):
    x = np.asarray(x)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    return np.max(np.abs(x))

surface_stats = (
    call_2.groupby(group_cols)
      .agg(
          n_strikes=('strike', 'size'),
          med_abs_d1=('iv_diff_1', median_),
          med_abs_d2=('iv_diff_2', median_),
          max_abs_d2=('iv_diff_2', max_)
      )
      .reset_index()
)

## 2a- Apply thresholds and flag noisy slices
- max_abs_d2 > 0.30: second-difference spike, likely a data error or stale quote
- n_strikes < 5: too few points to assess smoothness at all

In [7]:
SMOOTH_MAX_D2 = 0.30   
SMOOTH_MIN_N  = 5      

bad_slices = surface_stats[
    (surface_stats['max_abs_d2'] > SMOOTH_MAX_D2) |
    (surface_stats['n_strikes']  < SMOOTH_MIN_N)
][['quote_date', 'expiration']].copy()

before = len(call_2)
call_2 = call_2.merge(
    bad_slices.assign(_drop=True),
    on=['quote_date', 'expiration'],
    how='left'
)
call_2 = call_2[call_2['_drop'].isna()].drop(columns='_drop')
after = len(call_2)

print(f'Slices flagged : {len(bad_slices):,}')
print(f'Rows removed   : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining : {after:,}')

Slices flagged : 6,759
Rows removed   : 825,921  (8.29%)
Rows remaining : 9,141,126


## 2b- Quadratic-fit residual filter
- RMSE > 0.05 IV points on a quadratic in delta = the slice has a local kink
- (e.g., stale LEAPS quote, illiquid deep-OTM strike with erroneous IV)

In [8]:
from sklearn.linear_model import LinearRegression

def fit_slice_residuals(g):
    g = g.dropna(subset=['delta', 'implied_volatility']).sort_values('delta').copy()
    if len(g) < 5:
        g['iv_fit']   = np.nan
        g['iv_resid'] = np.nan
        return g
    X = np.column_stack([g['delta'].to_numpy(), g['delta'].to_numpy() ** 2])
    y = g['implied_volatility'].to_numpy()
    g['iv_fit']   = LinearRegression().fit(X, y).predict(X)
    g['iv_resid'] = y - g['iv_fit']
    return g

df_fit = (
    call_2.groupby(['quote_date', 'expiration'], group_keys=False)
          .apply(fit_slice_residuals, include_groups=False)
          .reset_index(drop=True)
)

# re-attach group keys
df_fit[['quote_date', 'expiration']] = call_2[['quote_date', 'expiration']].values

slice_quality = (
    df_fit.groupby(['quote_date', 'expiration'])
          .agg(
              n             = ('strike',   'size'),
              rmse_resid    = ('iv_resid', lambda x: np.sqrt(np.nanmean(x**2))),
              max_abs_resid = ('iv_resid', lambda x: np.nanmax(np.abs(x)))
          )
          .reset_index()
)

# inspect the distribution BEFORE choosing a cutoff
display(slice_quality['rmse_resid'].describe(percentiles=[.90, .95, .99]))
print(f"\nSlices above 0.05 : {(slice_quality['rmse_resid'] > 0.05).sum():,}")
print(f"Slices above 0.10 : {(slice_quality['rmse_resid'] > 0.10).sum():,}")
print(f"Slices above 0.15 : {(slice_quality['rmse_resid'] > 0.15).sum():,}")
print(f"Slices above 0.20 : {(slice_quality['rmse_resid'] > 0.20).sum():,}")

count    87928.000000
mean         0.077247
std          0.114792
min          0.000000
50%          0.043163
90%          0.173044
95%          0.269480
99%          0.601008
max          1.943802
Name: rmse_resid, dtype: float64


Slices above 0.05 : 37,109
Slices above 0.10 : 16,633
Slices above 0.15 : 10,615
Slices above 0.20 : 7,134


In [9]:
RESID_RMSE_MAX = 0.2

bad_fit = slice_quality[
    slice_quality['rmse_resid'] > RESID_RMSE_MAX
][['quote_date', 'expiration']]

before = len(call_2)
call_2 = call_2.merge(bad_fit.assign(_drop=True), on=['quote_date', 'expiration'], how='left')
call_2 = call_2[call_2['_drop'].isna()].drop(columns='_drop')
after  = len(call_2)

print(f'High-residual slices : {len(bad_fit):,}')
print(f'Rows removed         : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining       : {after:,}')

High-residual slices : 7,134
Rows removed         : 882,020  (9.65%)
Rows remaining       : 8,259,106


## 3- Smoothness across maturity (term structure)
- For each quote_date, ATM total variance w = σ²·T must be monotone non-decreasing in T
- A drop in total variance going from shorter to longer maturity is a structural data issue

In [10]:
call_3 = call_2.copy()

# ATM proxy: delta in [0.45, 0.55]
ATM_DELTA_LO, ATM_DELTA_HI = 0.45, 0.55

atm = call_3[call_3['delta'].between(ATM_DELTA_LO, ATM_DELTA_HI)].copy()
atm['T']         = (atm['expiration'] - atm['quote_date']).dt.days / 365.0
atm['total_var'] = atm['implied_volatility'] ** 2 * atm['T']

# Median total_var and T per (quote_date, expiration) across ATM strikes
atm_ts = (
    atm.groupby(['quote_date', 'expiration'])
       .agg(
           atm_iv    = ('implied_volatility', 'median'),
           total_var = ('total_var',          'median'),
           T         = ('T',                  'median')
       )
       .reset_index()
       .sort_values(['quote_date', 'T'])
)

# Flag: total_var decreases going from shorter to longer maturity
TV_MONOTONE_TOL = -0.002   # allow tiny dip (≤0.2% of total var) before flagging

atm_ts['tv_diff']      = atm_ts.groupby('quote_date')['total_var'].diff()
atm_ts['ts_violation'] = atm_ts['tv_diff'] < TV_MONOTONE_TOL

ts_violations = atm_ts[atm_ts['ts_violation']][['quote_date', 'expiration']]

print(f'Term-structure violations (non-monotone total var): {len(ts_violations):,}')
print(f'Affected quote dates                              : {ts_violations["quote_date"].nunique():,}')
display(atm_ts[atm_ts['ts_violation']][['quote_date','expiration','T','total_var','tv_diff']].head(10))

Term-structure violations (non-monotone total var): 37
Affected quote dates                              : 36


,quote_date,expiration,T,total_var,tv_diff
2015,2013-07-08,2015-01-17,1.528767,0.032994,-0.002497
3467,2013-11-21,2016-01-15,2.150685,0.040266,-0.004554
4123,2014-01-24,2016-01-15,1.975342,0.036983,-0.009490
4137,2014-01-27,2016-01-15,1.967123,0.036829,-0.004029
4249,2014-02-04,2014-09-30,0.652055,0.012208,-0.003060
4554,2014-02-27,2014-12-31,0.841096,0.011575,-0.003608
5193,2014-04-16,2014-12-31,0.709589,0.009765,-0.002956
5226,2014-04-21,2014-12-31,0.695890,0.009577,-0.002888
5525,2014-05-13,2016-01-15,1.676712,0.023074,-0.002957
6479,2014-07-29,2015-06-30,0.920548,0.012668,-0.004002


### Drop term-structure violations
- Remove only the offending expiration slices, not all slices on that date

In [11]:
before = len(call_3)
call_3 = call_3.merge(
    ts_violations.assign(_drop=True),
    on=['quote_date', 'expiration'],
    how='left'
)
call_3 = call_3[call_3['_drop'].isna()].drop(columns='_drop')
after = len(call_3)

print(f'Rows removed (TS violations) : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining               : {after:,}')

Rows removed (TS violations) : 3,035  (0.04%)
Rows remaining               : 8,256,071


## 4- Calendar spread arbitrage check
- For a fixed strike K, total implied variance w(K,T) = σ²(K,T)·T must be non-decreasing in T
- A decrease implies calendar spread arbitrage (longer-dated call cheaper than shorter-dated call at same K)

In [12]:
call_4 = call_3.copy()
call_4['T']         = (call_4['expiration'] - call_4['quote_date']).dt.days / 365.0
call_4['total_var'] = call_4['implied_volatility'] ** 2 * call_4['T']

call_4 = call_4.sort_values(['quote_date', 'strike', 'T'])

# Diff in total_var across maturities for the same (date, strike)
call_4['tv_diff_T'] = (
    call_4.groupby(['quote_date', 'strike'])['total_var'].diff()
)

CAL_TOL = -0.001   # 0.1% tolerance for floating-point noise
call_4['cal_violation'] = call_4['tv_diff_T'] < CAL_TOL

n_viol  = call_4['cal_violation'].sum()
n_total = len(call_4)
print(f'Calendar-spread violations : {n_viol:,}  ({n_viol/n_total*100:.2f}% of rows)')

viol_slices = (
    call_4[call_4['cal_violation']]
    .groupby(['quote_date', 'expiration'])
    .size()
    .reset_index(name='n_viol_strikes')
)
print(f'Slices with ≥1 calendar violation : {len(viol_slices):,}')
display(viol_slices.nlargest(10, 'n_viol_strikes'))

Calendar-spread violations : 198,160  (2.40% of rows)
Slices with ≥1 calendar violation : 24,323


,quote_date,expiration,n_viol_strikes
10413,2018-09-10,2018-10-19,82
23826,2024-07-09,2024-10-18,79
12,2013-01-11,2013-03-16,78
15148,2020-12-04,2020-12-31,78
136,2013-05-23,2014-01-18,76
14674,2020-09-30,2020-11-20,72
15162,2020-12-07,2021-03-19,72
15228,2020-12-15,2021-01-15,72
110,2013-04-05,2013-06-28,69
10394,2018-09-07,2018-09-28,69


### Drop calendar violations
- Drop only the violating rows (specific expiration for a given strike), not the whole slice
- A stale quote on one expiration should not remove adjacent expirations

In [13]:
before = len(call_4)
call_4 = call_4[~call_4['cal_violation']].drop(columns=['cal_violation'])
after  = len(call_4)

print(f'Rows dropped (calendar arb) : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining              : {after:,}')

Rows dropped (calendar arb) : 198,160  (2.40%)
Rows remaining              : 8,057,911


## 5- Butterfly spread arbitrage check
- For a fixed (date, expiry) slice, call prices must be convex in strike: C(K-h) - 2·C(K) + C(K+h) ≥ 0
- Negative butterfly spread implies a negative risk-neutral density → static arbitrage
- Normalized by forward strike spacing h to make the threshold strike-scale invariant

In [14]:
call_5 = call_4.drop(columns=['tv_diff_T'], errors='ignore').copy()
call_5 = call_5.sort_values(['quote_date', 'expiration', 'strike'])

grp = ['quote_date', 'expiration']

# vectorized shift within each (date, expiry) group
mid_prev = call_5.groupby(grp)['mid'].shift(1)
mid_next = call_5.groupby(grp)['mid'].shift(-1)

k_prev   = call_5.groupby(grp)['strike'].shift(1)
k_next   = call_5.groupby(grp)['strike'].shift(-1)

# raw butterfly and forward strike spacing
bfly = mid_prev - 2 * call_5['mid'] + mid_next
h    = (k_next - call_5['strike']).replace(0, np.nan)  # guard against duplicate strikes

call_5['butterfly_norm'] = bfly / h

BFLY_TOL = -0.01   # normalized threshold: >$0.01 violation per $1 of strike spacing

call_5['bfly_violation'] = call_5['butterfly_norm'] < BFLY_TOL

# interior points only — boundary NaNs (first/last per slice) are not violations
call_5.loc[call_5['butterfly_norm'].isna(), 'bfly_violation'] = False

n_viol = call_5['bfly_violation'].sum()
print(f'Butterfly violations : {n_viol:,}  ({n_viol/len(call_5)*100:.2f}% of rows)')

bfly_summary = (
    call_5.groupby(grp)
          .agg(
              n_viol          = ('bfly_violation',  'sum'),
              worst_butterfly = ('butterfly_norm',  'min')
          )
          .query('n_viol > 0')
          .reset_index()
          .sort_values('worst_butterfly')
)
print(f'Slices with ≥1 butterfly violation : {len(bfly_summary):,}')
display(bfly_summary.head(10))

Butterfly violations : 768,445  (9.54% of rows)
Slices with ≥1 butterfly violation : 75,066


,quote_date,expiration,n_viol,worst_butterfly
3420,2013-12-30,2013-12-31,8,-38.115
3328,2013-12-19,2013-12-31,8,-35.780
3226,2013-12-09,2013-12-31,6,-35.735
3131,2013-11-27,2013-12-31,6,-35.375
2409,2013-09-18,2013-09-30,5,-35.300
3148,2013-11-29,2013-12-31,6,-35.300
3091,2013-11-22,2013-12-31,10,-35.040
3162,2013-12-02,2013-12-31,8,-34.860
3074,2013-11-21,2013-12-31,10,-34.215
3177,2013-12-03,2013-12-31,6,-34.065


### Drop butterfly violations
- Drop only the individual rows where the butterfly condition is violated

In [15]:
before = len(call_5)
call_5 = call_5[~call_5['bfly_violation']].drop(columns=['butterfly_norm', 'bfly_violation'])
after  = len(call_5)

print(f'Rows dropped (butterfly arb) : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Rows remaining               : {after:,}')

Rows dropped (butterfly arb) : 768,445  (9.54%)
Rows remaining               : 7,289,466


## 6- Hard sanity bounds on Greeks and spreads
- delta ∈ [0, 1] inclusive: delta==1.0 are valid deep ITM calls (682K rows), not errors
- implied_volatility ∈ [0.01, 5.0]: 1% to 500% annualised
- gamma ≥ 0, vega ≥ 0: always non-negative for vanilla calls
- theta ≤ 0.01: small positive tolerance for near-expiry numerical noise (10K rows)
- rel_spread < 2.0: ask/bid spread cannot exceed 200% of mid

In [16]:
call_6 = call_5.copy()
before = len(call_6)

mask = (
    call_6['delta'].between(0.0, 1.0, inclusive='both') &   # was 'neither' — kept boundary values
    call_6['implied_volatility'].between(0.01, 5.0) &
    (call_6['gamma']      >= 0) &
    (call_6['vega']       >= 0) &
    (call_6['theta']      <= 0.01) &                        # was <= 0 — small noise tolerance
    (call_6['rel_spread'] < 2.0)
)

call_6 = call_6[mask].copy()
after  = len(call_6)

print(f'Rows dropped (sanity bounds) : {before - after:,}  ({(before-after)/before*100:.2f}%)')
print(f'Clean dataset                : {after:,} rows')

Rows dropped (sanity bounds) : 45,783  (0.63%)
Clean dataset                : 7,243,683 rows


## 7- Duplicate row check
- True duplicates: same (quote_date, expiration, strike) on the same day
- Soft duplicates: same key but different IV, keep the one closer to mid-market (lower rel_spread)

In [17]:
call_7 = call_6.copy() 

key_cols = ['quote_date', 'expiration', 'strike']

n_total        = len(call_7)
n_dupes        = call_7.duplicated(subset=key_cols, keep=False).sum()
n_unique_duped = call_7.duplicated(subset=key_cols, keep='first').sum()

print(f"Total rows              : {n_total:,}")
print(f"Rows involved in dupes  : {n_dupes:,}  ({n_dupes/n_total*100:.2f}%)")

Total rows              : 7,243,683
Rows involved in dupes  : 818  (0.01%)


### Resolve duplicates
- Keep the row with the lowest rel_spread (tightest market = most reliable quote)
- If rel_spread is tied, keep the first occurrence

In [18]:
before = len(call_7)

call_7 = (
    call_7
    .sort_values(key_cols + ['rel_spread'], ascending=True)
    .drop_duplicates(subset=key_cols, keep='first')
    .sort_values(key_cols)
    .reset_index(drop=True)
)

after = len(call_7)
print(f"Rows removed (duplicates) : {before - after:,}  ({(before-after)/before*100:.2f}%)")
print(f"Rows remaining            : {after:,}")

Rows removed (duplicates) : 409  (0.01%)
Rows remaining            : 7,243,274


## 8- Removing gaps in trading days
- gap > 5: contract not quoted for 6+ consecutive calendar days → stale, drop
- gap NaN: first observation per contract → keep (needed as iv_lag anchor)
- gap 3–5: weekends, long weekends, weekly-quoted LEAPS → all legitimate, keep

In [19]:
call_8 = call_7.copy()

before = len(call_8)

call_8['day_gap'] = (
    call_8['quote_date'] - call_8.groupby(['strike', 'expiration'])['quote_date'].shift(1)
).dt.days

call_8 = call_8[(call_8['day_gap'] <= 5) | call_8['day_gap'].isna()]

after = len(call_8)
print(f"Rows dropped (day_gap > 5) : {before - after:,}  ({(before-after)/before*100:.2f}%)")
print(f"Rows remaining             : {after:,}")

print(f"\nKept breakdown:")
print(f"  gap NaN (first obs) : {call_8['day_gap'].isna().sum():,}")
print(f"  gap 1               : {(call_8['day_gap'] == 1).sum():,}")
print(f"  gap 2               : {(call_8['day_gap'] == 2).sum():,}")
print(f"  gap 3               : {(call_8['day_gap'] == 3).sum():,}")
print(f"  gap 4               : {(call_8['day_gap'] == 4).sum():,}")
print(f"  gap 5               : {(call_8['day_gap'] == 5).sum():,}")

Rows dropped (day_gap > 5) : 173,662  (2.40%)
Rows remaining             : 7,069,612

Kept breakdown:
  gap NaN (first obs) : 225,974
  gap 1               : 4,661,278
  gap 2               : 481,913
  gap 3               : 1,148,158
  gap 4               : 411,141
  gap 5               : 141,148


## 9- Cleaning audit

In [20]:
call_raw = 11_012_971
call_0_n = 10_942_237

stages = {
    'raw calls'               : call_raw,
    'after NaN removal'       : call_0_n,
    'after basic filters'     : len(call_1),
    'after strike smoothness' : len(call_2),
    'after TS smoothness'     : len(call_3),
    'after calendar arb'      : len(call_4),
    'after butterfly arb'     : len(call_5),
    'after sanity bounds'     : len(call_6),
    'after duplicates'        : len(call_7),
    'after day gap filter'    : len(call_8),
}

prev = None
print(f'{"Stage":<30}  {"Rows":>10}  {"Dropped":>10}  {"% kept":>8}')
print('-' * 64)
for name, n in stages.items():
    dropped = f'{prev - n:>10,}' if prev is not None else f'{"—":>10}'
    pct     = f'{n/call_raw*100:>7.1f}%' if prev is not None else f'{"100.0%":>8}'
    print(f'{name:<30}  {n:>10,}  {dropped}  {pct}')
    prev = n

Stage                                 Rows     Dropped    % kept
----------------------------------------------------------------
raw calls                       11,012,971           —    100.0%
after NaN removal               10,942,237      70,734     99.4%
after basic filters              9,967,047     975,190     90.5%
after strike smoothness          8,259,106   1,707,941     75.0%
after TS smoothness              8,256,071       3,035     75.0%
after calendar arb               8,057,911     198,160     73.2%
after butterfly arb              7,289,466     768,445     66.2%
after sanity bounds              7,243,683      45,783     65.8%
after duplicates                 7,243,274         409     65.8%
after day gap filter             7,069,612     173,662     64.2%


# 10- Rename & save

In [21]:
print(call_8.columns.to_list())

['quote_date', 'symbol', 'expiration', 'type', 'strike', 'last', 'bid', 'ask', 'change', 'percent_change', 'volume', 'open_interest', 'implied_volatility', 'delta', 'gamma', 'theta', 'vega', 'rho', 'mid', 'spread', 'rel_spread', 'iv_diff_1', 'iv_diff_2', 'T', 'total_var', 'day_gap']


In [25]:
call_final = call_8.copy()

call_final = call_final.rename(columns={
    'quote_date':'date',
    'implied_volatility':'iv',
    'strike': 'k'
})

col_final = [
    'date', 'expiration', 'T', 'k', 'iv', 'mid', 'spread', 'volume', 
    'open_interest', 'delta', 'gamma', 'theta', 'vega', 'rho'
]

call_final = call_final[col_final]
call_final

,date,expiration,T,k,iv,mid,spread,volume,open_interest,delta,gamma,theta,vega,rho
0,2013-01-02,2013-01-04,0.005479,120.0,0.18930,26.050,0.22,10,140,1.0,0.0,-0.3634,0.0,-1.0
1,2013-01-02,2013-01-04,0.005479,121.0,0.18930,25.050,0.22,10,0,1.0,0.0,-0.3664,0.0,-1.0
2,2013-01-02,2013-01-04,0.005479,122.0,0.18930,24.050,0.22,0,0,1.0,0.0,-0.3694,0.0,-1.0
3,2013-01-02,2013-01-04,0.005479,123.0,0.18930,23.050,0.22,0,10,1.0,0.0,-0.3724,0.0,-1.0
4,2013-01-02,2013-01-04,0.005479,124.0,0.18930,22.050,0.22,0,0,1.0,0.0,-0.3755,0.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7243269,2026-03-04,2028-12-15,2.786301,1080.0,0.12745,3.500,5.00,0,1,0.06032,0.00082,-0.01239,1.36752,1.05322
7243270,2026-03-04,2028-12-15,2.786301,1090.0,0.12613,3.000,5.00,0,11,0.05323,0.00075,-0.01106,1.23923,0.93189
7243271,2026-03-04,2028-12-15,2.786301,1160.0,0.13814,2.505,4.99,0,0,0.04228,0.00057,-0.00967,1.03041,0.73675
7243272,2026-03-04,2028-12-15,2.786301,1180.0,0.14027,2.285,4.55,0,13,0.03852,0.00052,-0.00902,0.95545,0.67125


In [26]:
# flaot 64 to float 32
float64_cols = call_final.select_dtypes(include=['float64']).columns
call_final[float64_cols] = call_final[float64_cols].astype('float32')

call_final.to_parquet(INTERIM_DATA / '02-data-spy-option.parquet', index=False)